In [ ]:
import sys
from pathlib import Path
project_root = Path(__file__).resolve().parents[0] if '__file__' in globals() else Path().resolve().parents[0]
sys.path.insert(0, str(project_root))

### Make Encodings

In [ ]:
from tomato.utils import load_critic_review_df, tomato_data_path
df_full = load_critic_review_df()
df_unif = df_full\
    [~df_full.review_content.isna()]\
    .sample(1000,random_state=42)

In [ ]:
from tomato.encoding import bert_encode_reviews

texts = df_unif.review_content.astype(str).tolist()
ids = df_unif.index.tolist()
df_enc = bert_encode_reviews(texts, ids, "bert-base-uncased")

In [16]:
df_enc.to_parquet(tomato_data_path() / 'encoding_unif5k.parquet', compression='snappy')

### Test Pooling

In [ ]:
from tomato.utils import tomato_data_path
import pandas as pd

df_enc = pd.read_parquet(tomato_data_path() / 'encoding_unif5k.parquet')

In [ ]:
dim_cols = [c for c in df_enc.columns if c.startswith('dim_')]

In [ ]:
cls_vectors = df_enc[df_enc['token_id'] == 0][dim_cols].values

In [ ]:
import numpy as np
mean_vectors = np.vstack(
    df_enc\
        .groupby('review_id')\
        .apply(lambda g: np.average(
            g[dim_cols].values, weights=g.attention_mask, axis=0))\
        .values)

In [ ]:
from tomato.encoding import subtract_pcs

X0 = subtract_pcs(cls_vectors, 0, -1)
X = X0[0:100,:]


In [ ]:
from ripser import ripser
from persim import plot_diagrams

X = mean_vectors

results = ripser(X,2,distance_matrix=False)
diagrams = results['dgms']
plot_diagrams(diagrams, show=True)

### Test Metrics

In [ ]:
from tomato.metrics import pdist2
D = pdist2(X,X,"cosine")

In [ ]:
x = df_enc.groupby('review_id')
x.groups

In [ ]:
dim_cols = [c for c in df_enc.columns if c.startswith('dim_')]
group_dstrbs = [group[dim_cols].values for _, group in df_enc.groupby('review_id')]

In [ ]:
from tomato.metrics import wasserstein_distance
wasserstein_distance(group_dstrbs[0],group_dstrbs[1])

In [ ]:

pdist2(group_dstrbs[0],group_dstrbs[1]).shape

In [ ]:
import ot

X,Y = group_dstrbs[0],group_dstrbs[1]
n, m = X.shape[0], Y.shape[0]
# Cost matrix
C = pdist2(X, Y)
a = np.ones(n) / n
b = np.ones(m) / m
ot.emd2(a, b, C)

In [ ]:
len(group_dstrbs)

In [ ]:
from tomato.metrics import wasserstein_distances

df_enc_small = df_enc[df_enc.review_id.isin(df_enc.review_id.drop_duplicates().head(100))]
D = wasserstein_distances(df_enc_small)

In [ ]:
from tomato.metrics import wasserstein_distances_parallel
D = wasserstein_distances_parallel(df_enc)

In [ ]:
D = wasserstein_distances_parallel(df_enc_small)